#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "CID": "customer_key",
    "BDATE": "birth_date",
    "GEN": "gender"
}

#Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")
df.display()

# Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Null handling and Empty values

In [0]:
def is_empty(col):
    return col.isNull() | (F.trim(col) == "")

condition_invalid = (
    is_empty(F.col("CID")) |
    is_empty(F.col("BDATE")) |
    is_empty(F.col("GEN")) 
)

df_valid = df.filter(~condition_invalid)
df_invalid = df.filter(condition_invalid)

invalid_count = df_invalid.count()

print("Total rows:", df.count())
print("Invalid rows:", df_invalid.count())
print("Valid rows:", df_valid.count())

df = df_valid

## Data type casting

In [0]:
df = df.withColumn("BDATE", F.to_date("BDATE"))

## Handle duplicate ids

In [0]:
duplicate_cid_count = (
    df.groupBy("CID")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

print("Duplicate CID:", duplicate_cid_count)

window = Window.partitionBy("CID").orderBy(F.col("BDATE").desc())

df = (
    df.withColumn("row_num", F.row_number().over(window))
      .filter(F.col("row_num") == 1)
      .drop("row_num")
)

duplicate_after = (
    df.groupBy("CID")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

print("Duplicate CID after cleaning:", duplicate_after)


##Date validation

In [0]:
invalid_condition = (
    F.col("BDATE") > F.current_date()
)

df_invalid = df.filter(invalid_condition)
df_valid = df.filter(~invalid_condition)

total_count = df.count()
invalid_count = df_invalid.count()
valid_count = total_count - invalid_count

print("Total rows:", total_count)
print("Invalid future birthdates:", invalid_count)
print("Valid rows after cleaning:", valid_count)

null_count = df.filter(F.col("BDATE").isNull()).count()
print("Null birthdates:", null_count)

df = df_valid

##Normalization

In [0]:
df = df.withColumn(
    "GEN",
    F.when(F.upper(F.trim(F.col("GEN"))).isin("M", "MALE"), "Male")
     .when(F.upper(F.trim(F.col("GEN"))).isin("F", "FEMALE"), "Female")
     .otherwise("Unknown")
)

unknown_count = df.filter(F.col("GEN") == "Unknown").count()
print("Unknown gender count:", unknown_count)


## Renamig the columns

In [0]:
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks before write

In [0]:
def sanity_check(df):
    row_count = df.count()

    duplicate_customer_key = (
        df.groupBy("customer_key")
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    null_critical_fields = (
        df.filter(
            F.col("customer_key").isNull() |
            F.col("gender").isNull()
        )
        .count()
    )

    future_birthdate = (
        df.filter(
            F.col("birth_date") > F.current_date()
        )
        .count()
    )

    return {
        "row_count": row_count,
        "duplicate_customer_key": duplicate_customer_key,
        "null_critical_fields": null_critical_fields,
        "future_birthdate": future_birthdate
    }

results = sanity_check(df)
print("Before write:", results)

# Write Into Silver

In [0]:
(df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver.erp_customers"))

df_silver = spark.table("workspace.silver.erp_customers")

results = sanity_check(df_silver)
print("After write:", results)

if results["duplicate_customer_key"] > 0:
    raise Exception("Duplicate customer_key found!")

if results["null_critical_fields"] > 0:
    raise Exception("Null critical fields found!")

if results["future_birthdate"] > 0:
    raise Exception("Future birth_date values found!")
